# Scenario Evaluation — Manual Reassignment Tool

Explore the impact of manual labor→driver reassignments on schedule KPIs.

**Section 1 — Scenario Editor**  
Load a base solution, use the interactive panel to reassign labors to different drivers,
preview the impact, and save the result as a named scenario.

**Section 2 — Scenario Comparison**  
Compare two or more saved scenarios side-by-side using KPI tables, Gantt charts,
distance figures, and route maps.

> Both sections save to and read from the **same `scenarios/` directory** — no extra path configuration needed.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import List, Optional, Tuple

import pandas as pd
from IPython.display import display

# ── Project root on sys.path ────────────────────────────────────────────────
_PROJECT_ROOT = Path("../..").resolve()
if str(_PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / "src"))

# ── Scenario editor helpers ─────────────────────────────────────────────────
from alfred.analysis.scenario_editor import (
    ScenarioSet,
    apply_reassignments,
    build_labor_editor_table,
    build_multi_scenario_overview_table,
    build_service_display_info,
    load_driver_names,
    load_scenario_base,
    load_scenarios_for_comparison,
    reconstruct_scenario,
    save_scenario,
)

# ── Existing analysis utilities ─────────────────────────────────────────────
from alfred.analysis.compare_solutions import build_overview_table, load_and_prepare
from alfred.analysis.solution_evaluation import (
    build_driver_distance_figure,
    build_gantt_figure,
    build_route_map,
    build_service_distance_figure,
    compute_payload_summary,
)
from alfred.optimization.settings.solver_settings import DEFAULT_DISTANCE_METHOD

print("Imports OK — project root:", _PROJECT_ROOT)

Imports OK — project root: /Users/jbeta/Documents/AlfredProject/AlfredDEV


---
## Configuration

Set the paths to the experiment folder once here. Both sections will use the same `scenarios/` directory inside it.

In [2]:
# ── Paths — edit this cell ──────────────────────────────────────────────────
# Root of the experiment run (must contain output/ and api_snapshot/ sub-folders)
EXPERIMENT_DIR: Path = (
    _PROJECT_ROOT / "misc" / "experiments" / "scenarios" / "20260508"
)

# Derived paths (no edits needed below this line)
BASE_PAYLOAD:     Path          = EXPERIMENT_DIR / "output" / "output_payload.json"
DRIVER_DIRECTORY: Path          = EXPERIMENT_DIR / "api_snapshot" / "driver_directory_snapshot.json"
INPUT_FILE:       Optional[Path] = EXPERIMENT_DIR / "api_snapshot" / "optimization_input_snapshot.json"

# All scenarios (base copy + manual edits) land here
SCENARIOS_DIR: Path = EXPERIMENT_DIR / "scenarios"

# Filter to a single planning date (YYYY-MM-DD), or None to load all
PLANNING_DATE: Optional[str] = None

# Distance computation method — 'osrm' (default) or 'haversine'
DISTANCE_METHOD: str = DEFAULT_DISTANCE_METHOD

print(f"Experiment  : {EXPERIMENT_DIR}")
print(f"Base payload: {BASE_PAYLOAD}")
print(f"Scenarios   : {SCENARIOS_DIR}")
print(f"Driver dir  : {DRIVER_DIRECTORY} — exists: {DRIVER_DIRECTORY.exists()}")
print(f"Input file  : {INPUT_FILE} — exists: {INPUT_FILE.exists() if INPUT_FILE else False}")

Experiment  : /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508
Base payload: /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/output/output_payload.json
Scenarios   : /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/scenarios
Driver dir  : /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/api_snapshot/driver_directory_snapshot.json — exists: True
Input file  : /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/api_snapshot/optimization_input_snapshot.json — exists: True


---
## Section 1 — Scenario Editor

Run the cells in order:
1. **Load** the base solution
2. **Reassign** labors using the interactive panel
3. **Save** the result as a new scenario

In [3]:
# ── Load base solution ───────────────────────────────────────────────────────
_base = load_scenario_base(
    payload_path=BASE_PAYLOAD,
    driver_directory_path=DRIVER_DIRECTORY,
    planning_date=PLANNING_DATE,
    distance_method=DISTANCE_METHOD,
    input_file=INPUT_FILE,
)

# Load human-readable metadata for the widget UI
_driver_names = load_driver_names(DRIVER_DIRECTORY)   # {id_str: "First Last"}
_service_info = build_service_display_info(INPUT_FILE) # {labor_id: {labor_name, from_address, …}}

# Build the editor DataFrame (source of truth for the widget)
_editor_df = build_labor_editor_table(_base)

address_crosswalk_not_found path=/Users/jbeta/Documents/AlfredProject/AlfredDEV/data/examples/raw_files/address.csv


[driver_home_lookup] 12 drivers loaded
[coord_lookup] 26 labors | method='osrm'
[load_scenario_base] 21 labors | 9 drivers | 45 segments


In [4]:
# ── Interactive Reassignment Panel ───────────────────────────────────────────
import ipywidgets as W
from IPython.display import HTML, clear_output

# ── CSS ──────────────────────────────────────────────────────────────────────
display(HTML("""
<style>
.sc-card {
    display:flex; align-items:center; gap:12px;
    border:1px solid #e2e8f0; border-radius:8px;
    padding:9px 14px; margin-bottom:5px; background:#fff;
}
.sc-lnum  { font-size:11px; font-weight:700; color:#94a3b8; min-width:24px; text-align:right; }
.sc-lbody { flex:1; min-width:0; }
.sc-ltitle { font-size:13px; font-weight:600; color:#1e293b; }
.sc-lid    { font-weight:400; color:#94a3b8; font-size:11px; }
.sc-lsub   { font-size:11px; color:#64748b; margin-top:1px; }
.sc-badge  { display:inline-block; padding:1px 7px; border-radius:99px;
              font-size:10px; font-weight:600; margin-top:3px; }
.sc-badge-vt   { background:#dbeafe; color:#1d4ed8; }
.sc-badge-shop { background:#f0fdf4; color:#15803d; }
.sc-summary-box {
    background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px;
    padding:10px 16px; font-size:12px; margin-top:4px;
}
.sc-chg-row { display:flex; gap:8px; align-items:baseline;
               padding:3px 0; border-bottom:1px solid #f1f5f9; font-size:12px; }
.sc-chg-svc   { font-weight:600; color:#1e293b; min-width:90px; }
.sc-chg-labor { color:#475569; min-width:180px; }
.sc-chg-arrow { color:#94a3b8; }
.sc-chg-new   { font-weight:600; color:#0369a1; }
.sc-chg-old   { color:#94a3b8; font-size:10px; }
.sc-infeasible { color:#dc2626; font-weight:600; }
.sc-ok         { color:#16a34a; }
.sc-label-row  { font-size:11px; font-weight:700; text-transform:uppercase;
                  letter-spacing:.8px; color:#94a3b8; margin-bottom:4px; margin-top:2px; }
</style>
"""))

# ── Helper functions ─────────────────────────────────────────────────────────

def _driver_label(driver_id) -> str:
    if driver_id is None:
        return "— unassigned —"
    name = _driver_names.get(str(driver_id), "")
    return f"{name}  ({driver_id})" if name else str(driver_id)

def _svc_info_for(labor_id):
    return _service_info.get(labor_id) or _service_info.get(str(labor_id)) or {}

def _badge(labor_type: str) -> str:
    vt = {"alfred_initial_transport", "alfred_transport"}
    cls, txt = ("sc-badge-vt", "transport") if labor_type in vt else ("sc-badge-shop", "shop")
    return f'<span class="sc-badge {cls}">{txt}</span>'

# ── Build data structures ─────────────────────────────────────────────────────

# Sorted driver options (alphabetically by name)
_all_driver_ids = sorted(
    {str(r["driver_id"]) for r in _base.rows if r["driver_id"] is not None}
    | set(_driver_names.keys()),
    key=lambda d: _driver_names.get(d, d),
)
_driver_options = [(f"{_driver_names.get(d, d)}  ({d})", d) for d in _all_driver_ids]

# Group editor_df by service; preserve chronological order
_service_order = list(dict.fromkeys(_editor_df["service_id"].tolist()))
_rows_by_svc   = {sid: grp.to_dict("records")
                  for sid, grp in _editor_df.groupby("service_id", sort=False)}

# Original assignment map (immutable reference)
_orig_map = dict(zip(_editor_df["labor_id"], _editor_df["original_driver"]))

# One Dropdown per labor (created once, reused when switching services)
_dropdowns: dict = {}
for _, row in _editor_df.iterrows():
    orig = str(row["original_driver"]) if row["original_driver"] is not None else None
    _dropdowns[row["labor_id"]] = W.Dropdown(
        options=_driver_options,
        value=orig,
        layout=W.Layout(width="270px"),
    )

# ── Service selector ─────────────────────────────────────────────────────────

def _svc_option_label(svc_id) -> str:
    rows = _rows_by_svc.get(svc_id, [])
    n    = len(rows)
    info = _svc_info_for(rows[0]["labor_id"]) if rows else {}
    frm  = (info.get("from_address") or "")[:40]
    to   = (info.get("to_address")   or "")[:40]
    route = f"{frm}  →  {to}" if (frm or to) else f"Service {svc_id}"
    return f"{svc_id}  ·  {n} {'labor' if n == 1 else 'labors'}  ·  {route}"

_svc_dropdown = W.Dropdown(
    options=[(f"{_svc_option_label(sid)}", sid) for sid in _service_order],
    description="Service:",
    style={"description_width": "60px"},
    layout=W.Layout(width="840px"),
)

# ── Labor cards panel (rebuilt on each service change) ───────────────────────

_labor_area   = W.VBox(layout=W.Layout(margin="4px 0 0 0"))
_summary_area = W.Output()

def _build_labor_cards(svc_id):
    cards = []
    for row in _rows_by_svc.get(svc_id, []):
        lid   = row["labor_id"]
        lseq  = row.get("labor_sequence", 0)
        ltype = row.get("labor_type") or ""
        info  = _svc_info_for(lid)
        lname = info.get("labor_name") or ltype or "Labor"
        start = (row.get("actual_start") or "")[:16].replace("T", "  ")
        label_html = W.HTML(
            f'<div class="sc-card">'
            f'  <div class="sc-lnum">#{lseq + 1}</div>'
            f'  <div class="sc-lbody">'
            f'    <div class="sc-ltitle">'
            f'      {lname}'
            f'      <span class="sc-lid">(labor {lid})</span>'
            f'    </div>'
            f'    <div class="sc-lsub">{start}</div>'
            f'    {_badge(ltype)}'
            f'  </div>'
            f'</div>',
            layout=W.Layout(flex="1"),
        )
        dd = _dropdowns[lid]
        dd.observe(lambda _: _refresh_summary(), names="value")
        cards.append(W.HBox(
            [label_html, dd],
            layout=W.Layout(align_items="center", margin="0 0 2px 0"),
        ))
    return cards

def _refresh_summary():
    """Rebuild the live pending-changes table."""
    with _summary_area:
        clear_output(wait=True)
        changes = []
        for lid, dd in _dropdowns.items():
            orig_raw = _orig_map.get(lid)
            orig_str = str(orig_raw) if orig_raw is not None else None
            if dd.value != orig_str:
                changes.append((lid, orig_raw, dd.value))

        if not changes:
            display(HTML("<p style='color:#94a3b8;font-size:12px;margin:4px 0'>"
                         "No reassignments yet.</p>"))
            return

        rows_html = ""
        for lid, old_drv, new_drv in changes:
            row_match = _editor_df[_editor_df["labor_id"] == lid]
            svc_id = row_match.iloc[0]["service_id"] if len(row_match) else "—"
            lseq   = int(row_match.iloc[0].get("labor_sequence", 0)) if len(row_match) else 0
            info   = _svc_info_for(lid)
            lname  = info.get("labor_name") or str(lid)
            rows_html += (
                f"<div class='sc-chg-row'>"
                f"  <span class='sc-chg-svc'>Svc {svc_id}</span>"
                f"  <span class='sc-chg-labor'>#{lseq + 1} {lname}"
                f"    <span style='color:#b0bec5'>(#{lid})</span></span>"
                f"  <span class='sc-chg-arrow'>→</span>"
                f"  <span class='sc-chg-new'>{_driver_label(new_drv)}</span>"
                f"  <span class='sc-chg-old'>(was: {_driver_label(old_drv)})</span>"
                f"</div>"
            )
        display(HTML(
            f"<div class='sc-summary-box'>"
            f"  <div style='font-weight:700;color:#1e293b;margin-bottom:6px'>"
            f"    {len(changes)} pending reassignment(s)"
            f"  </div>"
            f"  {rows_html}"
            f"</div>"
        ))

def _on_service_change(change):
    _labor_area.children = _build_labor_cards(change["new"])

_svc_dropdown.observe(_on_service_change, names="value")
_labor_area.children = _build_labor_cards(_service_order[0])
_refresh_summary()

# ── Scenario name + Generate button ─────────────────────────────────────────

_name_input = W.Text(
    value="scenario_v1",
    placeholder="e.g. scenario_v1",
    description="Scenario name:",
    style={"description_width": "120px"},
    layout=W.Layout(width="320px"),
)
_generate_btn = W.Button(
    description=" Generate & Save",
    button_style="success",
    icon="check",
    layout=W.Layout(width="185px", height="36px"),
    tooltip="Reconstruct the schedule with all pending reassignments and save to scenarios/",
)
_result_output = W.Output()

# Cache used by Section 2 auto-append
_last_scenario: dict = {"rows": None, "segments": None, "label": None, "path": None}


def _on_generate(_btn):
    with _result_output:
        clear_output(wait=True)

        reassignments = {
            lid: dd.value
            for lid, dd in _dropdowns.items()
            if dd.value != (str(_orig_map.get(lid))
                            if _orig_map.get(lid) is not None else None)
        }
        display(HTML(
            f"<p style='color:#64748b;font-size:12px'>"
            f"Applying <b>{len(reassignments)}</b> reassignment(s)…</p>"
        ))

        new_rows, warnings = apply_reassignments(_base.rows, reassignments)
        if warnings:
            display(HTML("".join(
                f"<p style='color:#b45309;font-size:11px'>⚠ {w}</p>"
                for w in warnings
            )))

        new_rows, new_segs = reconstruct_scenario(
            new_rows, _base.points_lookup, _base.driver_home_lookup,
            _base.params, DISTANCE_METHOD,
        )

        # Infeasibility report
        infeasible = [r for r in new_rows if r.get("is_infeasible")]
        if infeasible:
            inf_rows = "".join(
                f"<tr>"
                f"<td style='padding:2px 10px 2px 0'>{r['labor_id']}</td>"
                f"<td style='padding:2px 10px 2px 0'>{r['service_id']}</td>"
                f"<td style='padding:2px 10px 2px 0'>{_driver_label(r['driver_id'])}</td>"
                f"<td>{str(r['actual_start'])[:16]}</td>"
                f"</tr>"
                for r in infeasible
            )
            display(HTML(
                f"<div class='sc-summary-box'>"
                f"<span class='sc-infeasible'>⚠ {len(infeasible)} infeasible labor(s)</span>"
                f"<table style='margin-top:6px;font-size:11px;border-collapse:collapse'>"
                f"<thead><tr style='color:#94a3b8'>"
                f"<th>Labor</th><th>Service</th><th>Driver</th><th>Start</th>"
                f"</tr></thead><tbody>{inf_rows}</tbody></table></div>"
            ))
        else:
            display(HTML("<p class='sc-ok' style='font-size:12px'>✓ No infeasible labors.</p>"))

        # Save
        label = _name_input.value.strip() or "scenario_v1"
        SCENARIOS_DIR.mkdir(parents=True, exist_ok=True)
        saved = save_scenario(
            _base.services, new_rows,
            SCENARIOS_DIR / f"{label}.json", label,
        )
        _last_scenario.update(rows=new_rows, segments=new_segs, label=label, path=saved)

        # KPI summary
        summary = compute_payload_summary(
            new_rows,
            tiempo_gracia_min=_base.params.tiempo_gracia_min,
            segments=new_segs,
        )
        kpis = [
            ("Labors assigned",  summary.get("labors_assigned", "—")),
            ("Infeasible",       summary.get("labors_infeasible", "—")),
            ("Total labor dist", f"{summary.get('total_labor_distance_km', 0):.1f} km"),
            ("Total move dist",  f"{summary.get('total_driver_move_distance_km', 0):.1f} km"),
        ]
        kpi_html = "".join(
            f"<tr><td style='padding:2px 14px 2px 0;color:#64748b'>{k}</td>"
            f"<td style='font-weight:600'>{v}</td></tr>"
            for k, v in kpis
        )
        display(HTML(
            f"<div class='sc-summary-box'>"
            f"  <b>✓ Scenario saved</b>"
            f"  <code style='font-size:11px;margin-left:6px'>{saved.name}</code>"
            f"  <table style='margin-top:8px;font-size:12px'>{kpi_html}</table>"
            f"</div>"
        ))


_generate_btn.on_click(_on_generate)

# ── Render ───────────────────────────────────────────────────────────────────
display(W.VBox([
    W.HTML("<div class='sc-label-row'>Select a service</div>"),
    _svc_dropdown,
    _labor_area,
    W.HTML("<hr style='margin:12px 0;border-color:#e2e8f0'>"),
    W.HTML("<div class='sc-label-row'>Pending reassignments</div>"),
    _summary_area,
    W.HTML("<hr style='margin:12px 0;border-color:#e2e8f0'>"),
    W.HBox([_name_input, _generate_btn],
           layout=W.Layout(gap="12px", align_items="center")),
    _result_output,
], layout=W.Layout(max_width="880px")))

---
## Section 2 — Scenario Comparison

All saved scenarios (including the base) are read from `SCENARIOS_DIR`.
The base payload is automatically copied there on first run so it can be
compared on equal footing with any edited scenario.

- The **first entry** in `SCENARIO_PATHS` is the baseline for Δ columns.
- Run this section independently by pointing `SCENARIOS_DIR` to any folder
  containing pre-saved scenario JSON files.

In [5]:
# ── Copy the base payload into the scenarios folder (once) ───────────────────
import shutil

SCENARIOS_DIR.mkdir(parents=True, exist_ok=True)
_base_copy = SCENARIOS_DIR / "base.json"
if not _base_copy.exists():
    shutil.copy2(BASE_PAYLOAD, _base_copy)
    print(f"Base payload copied → {_base_copy}")
else:
    print(f"Base already present: {_base_copy}")

# ── List all available scenarios in the folder ───────────────────────────────
_available = sorted(SCENARIOS_DIR.glob("*.json"))
print(f"\nAvailable scenarios in {SCENARIOS_DIR}:")
for p in _available:
    print(f"  {p.name}")

Base already present: /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/scenarios/base.json

Available scenarios in /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/scenarios:
  base.json
  scenario_v1.json


In [6]:
# ── Section 2 Configuration ──────────────────────────────────────────────────
# Scenarios to compare — (path, label) list; first entry = baseline.
# By default: compare base vs. the last scenario generated in Section 1
# (or list explicit paths from SCENARIOS_DIR if running Section 2 standalone).
SCENARIO_PATHS: List[Tuple[Path, str]] = [
    (SCENARIOS_DIR / "base.json",        "base"),
    # Add more:
    (SCENARIOS_DIR / "scenario_v1.json", "scenario_v1"),
    # (SCENARIOS_DIR / "scenario_v2.json", "scenario_v2"),
]

# If Section 1 was just run, append the new scenario automatically
if _last_scenario.get("path") and _last_scenario["path"] not in [p for p, _ in SCENARIO_PATHS]:
    SCENARIO_PATHS.append((_last_scenario["path"], _last_scenario["label"]))
    print(f"Auto-added last scenario: {_last_scenario['label']}")

print(f"Comparing: {[lbl for _, lbl in SCENARIO_PATHS]}")

Comparing: ['base', 'scenario_v1']


In [7]:
# ── Load all scenarios ───────────────────────────────────────────────────────
_scenario_set = load_scenarios_for_comparison(
    scenario_paths=SCENARIO_PATHS,
    driver_directory_path=DRIVER_DIRECTORY,
    planning_date=PLANNING_DATE,
    distance_method=DISTANCE_METHOD,
    input_file=INPUT_FILE,
)

[driver_home_lookup] 12 drivers loaded
[coord_lookup] 26 labors | method='osrm'

[load_scenarios] Loading 'base' from base.json
  21 labors | 9 drivers | 45 segments

[load_scenarios] Loading 'scenario_v1' from scenario_v1.json
  21 labors | 9 drivers | 44 segments


In [8]:
# ── KPI Overview Table (N scenarios) ────────────────────────────────────────
_overview = build_multi_scenario_overview_table(
    _scenario_set,
    baseline_label=SCENARIO_PATHS[0][1],
)
display(
    _overview.style
    .format(precision=2, na_rep="—")
    .set_caption(f"Scenario Comparison — Δ vs '{SCENARIO_PATHS[0][1]}'")
)

,base,scenario_v1,Δ scenario_v1 vs base
metric,,,
services,17.00,17.00,0.00
labors_total,21.00,21.00,0.00
labors_vt,19.00,19.00,0.00
labors_non_vt,2.00,2.00,0.00
drivers_used,9.00,9.00,0.00
labors_assigned,19.00,19.00,0.00
labors_infeasible,0.00,2.00,2.00
labors_infeasible_pct,0.00,9.52,9.52
labors_in_grace,0.00,0.00,0.00


In [9]:
# ── Pairwise comparison — runs only for exactly 2 scenarios ─────────────────
if len(SCENARIO_PATHS) == 2:
    _la, _lb = SCENARIO_PATHS[0][1], SCENARIO_PATHS[1][1]
    _pair = load_and_prepare(
        sol_a=SCENARIO_PATHS[0][0],
        sol_b=SCENARIO_PATHS[1][0],
        driver_directory=DRIVER_DIRECTORY,
        planning_date=PLANNING_DATE,
        distance_method=DISTANCE_METHOD,
        input_file=INPUT_FILE,
    )
    display(build_overview_table(_pair, _la, _lb))
else:
    print(f"Pairwise cell skipped ({len(SCENARIO_PATHS)} scenarios loaded).")

[driver_home_lookup] 12 drivers loaded
[coord_lookup] 26 labors | method='osrm'
sol_a: 21 labors | 9 drivers | 45 segments
sol_b: 21 labors | 9 drivers | 44 segments


,base,scenario_v1,delta,delta_pct
metric,,,,
services,17.00,17.00,0.00,0.00
labors_total,21.00,21.00,0.00,0.00
labors_vt,19.00,19.00,0.00,0.00
labors_non_vt,2.00,2.00,0.00,0.00
drivers_used,9.00,9.00,0.00,0.00
labors_assigned,19.00,19.00,0.00,0.00
labors_infeasible,0.00,2.00,2.00,NaN
labors_infeasible_pct,0.00,9.52,9.52,NaN
labors_in_grace,0.00,0.00,0.00,NaN


In [10]:
# ── Gantt Charts — one per scenario ─────────────────────────────────────────
for label, rows, segs in zip(
    _scenario_set.labels,
    _scenario_set.rows_list,
    _scenario_set.segments_list,
):
    _drivers = sorted({r["driver_id"] for r in rows if r["driver_id"] is not None})
    build_gantt_figure(segs, _drivers, label).show()

In [11]:
# ── Distance per Service — one chart per scenario ────────────────────────────
for label, rows in zip(_scenario_set.labels, _scenario_set.rows_list):
    build_service_distance_figure(rows, _scenario_set.all_services, label).show()

In [12]:
# ── Distance per Driver — one chart per scenario ─────────────────────────────
for label, rows in zip(_scenario_set.labels, _scenario_set.rows_list):
    _drivers = sorted({r["driver_id"] for r in rows if r["driver_id"] is not None})
    build_driver_distance_figure(rows, _drivers, label).show()

In [13]:
# ── Route Maps — driver dropdown, one map per scenario ───────────────────────
import ipywidgets as W
from IPython.display import HTML

_drv_options = [
    (f"{_driver_names.get(d, d)}  ({d})", d)
    for d in _scenario_set.all_drivers
]
_drv_dropdown = W.Dropdown(
    options=_drv_options,
    description="Driver:",
    layout=W.Layout(width="320px"),
)
_map_out = W.Output()


def _render_maps(change):
    _map_out.clear_output(wait=True)
    did = change["new"]
    with _map_out:
        for label, rows in zip(_scenario_set.labels, _scenario_set.rows_list):
            if not any(r["driver_id"] == str(did) for r in rows):
                display(W.HTML(f"<p style='color:#94a3b8;font-size:12px'><b>{label}</b>: driver {did!r} has no labors.</p>"))
                continue
            home_wkt = (_scenario_set.driver_home_lookup or {}).get(str(did))
            display(W.HTML(f"<h4 style='margin:12px 0 4px'>{label}</h4>"))
            try:
                fig = build_route_map(
                    services=None,
                    rows=rows,
                    driver_id=did,
                    driver_home_wkt=home_wkt,
                    label=label,
                    points_lookup=_scenario_set.points_lookup,
                )
                display(HTML(fig._repr_html_()))
            except Exception as exc:
                display(W.HTML(f"<p style='color:red'>Map error: {exc}</p>"))


_drv_dropdown.observe(_render_maps, names="value")
_render_maps({"new": _drv_dropdown.value})
display(_drv_dropdown, _map_out)

Dropdown(description='Driver:', layout=Layout(width='320px'), options=(('Ivan Dario Pinta  (10451)', '10451'),…

Output()